# 12 · A crowd heads for coffee 🚶☕

> **Note**
>
> This notebook is **best run locally or in the cloud** — the transport loop takes
> many small time steps and the closing *parallel* section needs MPI, neither of
> which suits the in-browser *JupyterLite* kernel. Two good ways to run it:
>
> - ▶ **Open in Google Colab** (badge above) — a full Linux VM; the **first cell
>   `pip install`s NGSolve** for you (about a minute on first launch), then it runs
>   like any local notebook, webgui included.
> - 🖥 Grab the `.py`/`.ipynb` from the header and run it with your own NGSolve.

So far everything was driven by *diffusion* (heat). The world is also full of
**transport**. Here is a charming one: the talk just ended, and the **audience** —
packed in behind the **rows of tables**, with a lone **presenter** still up front —
streams toward the single door to reach the **coffee break** ☕.
Pedestrian flow is modelled by a **conservation law** for the crowd density
$\rho\in[0,1]$ (1 = shoulder-to-shoulder):
$$ \partial_t \rho + \nabla\!\cdot\bigl(\rho\,v(\rho)\,\hat{\mathbf e}\bigr)=0,
   \qquad v(\rho)=v_{\max}\,(1-\rho). $$
Two ingredients make it *pedestrian*: people **slow down in a crowd**
($v(\rho)\to 0$ as $\rho\to 1$ — this nonlinearity is what creates **jams**), and
they walk in a **desired direction** $\hat{\mathbf e}$ toward the exit.

In [ ]:
# --- Google Colab: install NGSolve on first run (a no-op anywhere else) -------
# NGSolve ships its PyPI wheels as pre-releases, so the `--pre` flag is essential
# (without it pip finds no matching wheel and `netgen` ends up missing).
import sys
if "google.colab" in sys.modules:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "--pre",
                    "ngsolve", "webgui_jupyter_widgets"], check=True)

In [ ]:
from netgen.occ import WorkPlane, OCCGeometry
from ngsolve import *
from ngsolve.webgui import Draw
import sys

def progress(i, n, label="working"):
    """A tiny dependency-free progress bar (survives JupyterLite / Colab / local)."""
    import os
    if os.environ.get("WEBGUI_SCENE_DIR"):             # static-site build: stay silent (no \r spam in the HTML)
        return
    if (i + 1) % max(1, n // 100) == 0 or i + 1 == n:
        f = int(26 * (i + 1) / n)
        sys.stdout.write(f"\r  {label}… [{'█'*f}{'·'*(26-f)}] {100*(i+1)//n:3d}%")
        sys.stdout.flush()
        if i + 1 == n:
            sys.stdout.write("\n")

# the lecture hall: three staggered rows of tables (with alternating side aisles)
# and a single narrow door at the front-left, leading to the coffee corridor
W, H = 6.0, 4.5
hall = (WorkPlane().MoveTo(0, 0).LineTo(0.4, 0).LineTo(1.2, 0)   # door segment 0.4–1.2
        .LineTo(W, 0).LineTo(W, H).LineTo(0, H).Close().Face())
rows = [WorkPlane().MoveTo(1.5, 1.4).Rectangle(5.0, 0.3).Face(),    # aisle on the left
        WorkPlane().MoveTo(-0.5, 2.4).Rectangle(5.0, 0.3).Face(),   # aisle on the right
        WorkPlane().MoveTo(1.5, 3.4).Rectangle(5.0, 0.3).Face()]    # aisle on the left
for r in rows:
    hall = hall - r                                    # carve the tables out of the room
hall.edges.name = "wall"                               # tables count as walls (no-flux)
for e in hall.edges:                                   # name the door segment
    if abs(e.center[1]) < 1e-6 and abs(e.center[0] - 0.8) < 0.45:
        e.name = "exit"
mesh = Mesh(OCCGeometry(hall, dim=2).GenerateMesh(maxh=0.13))
print(f"{mesh.ne} elements; boundaries: {set(mesh.GetBoundaries())}")

## 1. The navigation field — which way is *out*?

People do not walk in a fixed compass direction; they head for the **door**, and
they **follow walls** rather than walking into them. A cheap, realistic way to get
such a field is a **potential**: solve a little Laplace problem with the door at
"low pressure" ($\varphi=0$) and the walls insulating ($\partial_n\varphi=0$), then
walk **downhill**, $\hat{\mathbf e}=-\nabla\varphi/|\nabla\varphi|$. The Neumann
wall condition makes $\hat{\mathbf e}$ automatically **tangential** at walls — so
nobody walks through a wall, for free. The **table rows are walls too**, so the
field threads everyone through the **aisles** and around the furniture.

In [ ]:
P = H1(mesh, order=2, dirichlet="exit")
u, v = P.TnT()
phi = GridFunction(P)
ap = BilinearForm(grad(u) * grad(v) * dx).Assemble()
fp = LinearForm(1 * v * dx).Assemble()
phi.vec.data = ap.mat.Inverse(P.FreeDofs(), inverse="sparsecholesky") * fp.vec

nav = GridFunction(VectorH1(mesh, order=1))            # a smooth (continuous) field
nav.Set(-grad(phi) / sqrt(grad(phi) * grad(phi) + 1e-3))
Draw(nav, mesh, "navigation field ê — everyone heads for the door",
     vectors={"grid_size": 28})

## 2. A higher-order discontinuous-Galerkin scheme

Transport keeps **sharp fronts sharp**, so **discontinuous Galerkin** is the
natural tool — and we want it **accurate**, so we use a genuine **higher order**,
`L2(mesh, order=2)`: each cell carries a *quadratic* density profile,
discontinuous across edges. Cells communicate only through a **numerical flux** on
shared edges; we use the robust **Lax–Friedrichs** flux
$$ \hat f = \tfrac12\bigl(f(\rho)+f(\rho^{\text{other}})\bigr)\!\cdot\!\mathbf n
           -\tfrac12\,v_{\max}\,(\rho^{\text{other}}-\rho), $$
with $f(\rho)=\rho\,v(\rho)\,\hat{\mathbf e}$. At the **door** the crowd flows out;
every other edge is a wall with **no flux**.

A high-order scheme **overshoots** at a steep front (Godunov's theorem — see the
quiz below), so we add a touch of **artificial viscosity**: an $h$-scaled
DG-Laplacian (interior penalty) that damps the wiggles while keeping the front far
sharper than low order would. This is the standard NGSolve recipe for stabilising
high-order conservation laws.

In [ ]:
order = 2
fes = L2(mesh, order=order, dgjumps=True)
rho, w = fes.TnT()
n = specialcf.normal(2)
vmax = 1.0
def flux(r): return vmax * r * (1 - r)                 # speed·density magnitude (along ê)

ro = rho.Other()
bn = nav * n
fhat = 0.5 * (flux(rho) + flux(ro)) * bn - 0.5 * vmax * (ro - rho)   # Lax–Friedrichs
conv = (-flux(rho) * (nav * grad(w)) * dx                           # volume term
        + fhat * (w - w.Other()) * dx(skeleton=True)               # interior edges
        + flux(rho) * bn * w * ds(skeleton=True, definedon=mesh.Boundaries("exit")))  # outflow
C = BilinearForm(conv, nonassemble=True)               # nonlinear — applied explicitly

# artificial viscosity: an h-scaled interior-penalty DG-Laplacian (the standard
# high-order stabiliser); it is linear, so we assemble it once.
h = specialcf.mesh_size
visc = 2.0 * 0.5 * h / order
def avg(s, t): return 0.5 * (s + t)
diff = (visc * grad(rho) * grad(w) * dx
        + visc * (-avg(grad(rho), grad(ro)) * n * (w - w.Other())
                  - avg(grad(w), grad(w.Other())) * n * (rho - ro)
                  + 10 * order**2 / h * (rho - ro) * (w - w.Other())) * dx(skeleton=True))
D = BilinearForm(diff).Assemble()
M = BilinearForm(rho * w * dx).Assemble()

## 3. IMEX time stepping — and watch the jam

The artificial-viscosity term is **stiff** (its interior penalty scales like
$1/h$), so an explicit step would demand an absurdly small $\Delta t$. The classic
cure is **IMEX**: treat the cheap **convection explicitly** and the stiff
**viscosity implicitly**. We factor $M+\Delta t\,D$ **once** (it is symmetric
positive definite → `sparsecholesky`) and reuse it every step,
$$ (M+\Delta t\,D)\,\rho^{n+1} = M\,\rho^{n} - \Delta t\,C(\rho^{n}). $$
The crowd fills the hall and streams out; the density **piles up at the door** —
as $\rho\to1$ people slow to a crawl (the bottleneck **jam**) — then drains.

In [ ]:
# initial crowd: the audience packed in behind the table rows (a smooth step in y),
# plus a lone presenter blob still up at the front
audience = 0.7 * 0.5 * (1 + (y - 2.0) / sqrt((y - 2.0)**2 + 0.08))
presenter = 0.85 * exp(-((x - 4.5)**2 + (y - 0.7)**2) / 0.12)
gfu = GridFunction(fes)
gfu.Set(audience + presenter)
mass0 = Integrate(gfu, mesh)

dt = 0.005
mstar = M.mat.CreateMatrix()                                 # M + dt·D, factored once
mstar.AsVector().data = M.mat.AsVector() + dt * D.mat.AsVector()
mstarinv = mstar.Inverse(inverse="sparsecholesky")           # SPD → Cholesky

res = gfu.vec.CreateVector(); rhs = gfu.vec.CreateVector()
gfu.AddMultiDimComponent(gfu.vec)                            # frame 0
with TaskManager():
    for step in range(1000):
        C.Apply(gfu.vec, res)                                # explicit convection
        rhs.data = M.mat * gfu.vec - dt * res
        gfu.vec.data = mstarinv * rhs                        # implicit viscosity
        if step % 30 == 0:
            gfu.AddMultiDimComponent(gfu.vec)
        progress(step, 1000, "streaming")

print(f"crowd in hall: {mass0:.2f} → {Integrate(gfu, mesh):.2f} "
      f"(the difference reached the coffee ☕)")
print(f"peak density {max(gfu.vec):.2f};  faint undershoot to {min(gfu.vec):.2f} "
      f"(the Gibbs whisker — see the quiz)")
Draw(gfu, mesh, interpolate_multidim=True, animate=True)

Three things are worth pausing on:

* **Mass is conserved** — every person leaving the hall is accounted for at the
  door; the conservative DG flux loses nobody in the interior.
* The high-order scheme resolves the jam front **sharply** — much crisper than a
  lowest-order (finite-volume) picture would be.
* Look closely behind the front and you will spot a faint **undershoot** — a
  whisker of "negative people". That is the unavoidable **Gibbs/Godunov** wiggle
  of any linear high-order scheme; the artificial viscosity keeps it to a few
  percent.

## 4. Going faster — parallelism

City-scale crowds or traffic are *huge*, so the next question is **speed**.
NGSolve parallelises on two levels.

**Shared memory (threads).** We already wrapped the loop in a `TaskManager` — the
flux applications and mass solves then run on all cores, no code change.

**Distributed memory (MPI).** For problems too big for one machine, NGSolve splits
the mesh across processes that exchange only their shared boundaries. The *same*
Python code runs under `mpirun`:

```python
# run with:  mpirun -np 4 python3 crowd_mpi.py
from mpi4py import MPI
from ngsolve import *
comm = MPI.COMM_WORLD
mesh = Mesh(unit_square.GenerateMesh(maxh=0.05, comm=comm))   # distributed mesh
print(f"rank {comm.rank} of {comm.size} owns {mesh.ne} elements")
# ... assemble & step exactly as above — NGSolve handles the communication ...
```

## Where to go from here

This was a deliberately small taste. The official NGSolve **i-tutorials** go much
deeper:

- **Unit 2.8** — discontinuous Galerkin and hybrid DG in full,
- **Unit 3.x** — time-dependent problems, DG operator application, limiters,
- **Unit 5a** — MPI-parallel NGSolve from the ground up,
- and the closing **outlook unit** of this course.

:::{dropdown} 🧠 Final quiz — when would you reach for DG instead of plain H1?
Whenever the solution is **transport-dominated** or genuinely discontinuous
(shocks, sharp fronts, conservation laws), when you need a **locally conservative**
method, or when you want **cheap explicit time stepping** thanks to the
block-diagonal mass matrix. For smooth, diffusion-dominated problems — like the
heat equation in notebooks 9 and 11 — continuous $H^1$ elements are simpler and
usually cheaper. Pick the tool that matches the physics. ☕
:::

In [ ]:
# Navigation between units — shown only in a live notebook (Colab / JupyterLite /
# local Jupyter), never in the rendered website (which has its own prev/next nav).
import os, sys
if not os.environ.get("WEBGUI_SCENE_DIR"):          # not the static site build
    _prev = ("11-turing-patterns", "11 · How the beast got its stripes 🌈")
    _next = ("13-thermal-plume-hdg", "13 · A puffing thermal plume — HDiv-HDG & HDG 🔥🌀")
    def _u(_nb):
        if "google.colab" in sys.modules:
            return "https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/" + _nb + ".ipynb"
        return _nb + ".ipynb"                       # JupyterLite & local: relative .ipynb link
    _parts  = ["⬅️ **Previous:** [%s](%s)" % (_prev[1], _u(_prev[0]))] if _prev else []
    _parts += ["➡️ **Next:** [%s](%s)" % (_next[1], _u(_next[0]))] if _next else []
    from IPython.display import display, Markdown
    display(Markdown(" · ".join(_parts)))